# Phase 4 CJPE Stage 2 — V2 Improved Fine-Tuning Candidate

## Objective

This notebook adapts the supplied Phase-1 V2 encoder-decoder checkpoint for binary court-judgment prediction. The original V2 architecture, pretrained checkpoint, and matching BPE-25k tokenizer are preserved. The autoregressive decoder is loaded strictly for checkpoint compatibility, then omitted from the Phase-4 classification path because classification uses the source encoder.

## Why this revision exists

The previous run reported **development Macro-F1 = 0.5838** at epoch 2 and used four epochs with only the LoRMA+ matrices and a linear classification head trainable. It sampled one window per training case per epoch while scoring up to eight windows at inference. That difference can create a weak training signal when the decisive evidence is not present in the sampled window.

This revision tests a reasoned, more expressive adaptation strategy:

1. **Train on two windows per case** and average their logits before calculating the case-level loss. This better aligns training with the document-level inference rule and reduces the chance that a single arbitrary window determines the gradient.
2. **Sample windows reproducibly and differently each epoch**, without replacement when a document has enough windows, so longer cases receive broader training coverage.
3. **Increase LoRMA+ rank from 8 to 16**, increasing adapter capacity while keeping the base Q/V projections frozen.
4. **Unfreeze only the final encoder block and encoder final normalization**, using a smaller learning rate for pretrained weights than for new parameters. This creates a middle ground between an almost fully frozen backbone and full fine-tuning.
5. **Combine masked mean pooling with learned attention pooling**, allowing the classifier to use both a broad representation and a learned summary of salient tokens.
6. **Train for up to eight epochs with early stopping**, selecting the checkpoint by development Macro-F1 at threshold 0.50. More epochs are a ceiling, not a requirement.
7. **Add executable smoke tests** for output shapes, padding-mask invariance, gradient flow, and finite gradients before training.

These are hypotheses to evaluate, not guaranteed improvements. The development set chooses checkpoints and the final decision threshold; the test set is never used for model selection. The target of 0.70+ Macro-F1 is aspirational, not guaranteed.

## Experiment integrity

- The Phase-1 V2 architecture, pretrained checkpoint tensors, tokenizer, data split, labels, and 512-token context length remain unchanged.
- Changes are limited to Phase-4 adaptation, training-window sampling, pooling, optimizer groups, training duration, and diagnostics.
- The original notebook is retained separately; this is a new candidate notebook.
- The `revision_notes` field in `development_metrics.json` records the effective configuration. Use the report `V2_Notebook_Change_Design_and_QA_Report.docx` for the full rationale and QA checklist.

References: Bihany, Patel & Modi, “LoRMA: Low-Rank Multiplicative Adaptation for LLMs,” Findings of ACL 2025, https://aclanthology.org/2025.findings-acl.527/ . The classification-specific changes above are engineering hypotheses that must be judged on held-out development performance.

**Run order:** attach `train.json`, `dev.json`, `test.json`, `v2_model.pt`, and `bpe_en_25k.json` exactly once each to Kaggle, select a GPU accelerator, restart the session, and select **Run All** from the first cell.


### Local Windows Porting Notes & Environment Adaptations

This notebook is `Phase4_CJPE_V2_Improved_Local.ipynb`, adapted from `Phase4_CJPE_V2_Improved.ipynb` for local execution on Windows with an NVIDIA RTX 4060 (8 GB VRAM):
1. **Local Paths & Asset Resolution:** Replaced `/kaggle/input` and `/kaggle/working` with configurable paths defaulting to `D:\CJPE_Phase4_V2_Local` and `D:\CJPE_Phase4_V2_Improved_Local\outputs\phase4_v2_improved`. Resolved all 5 required files (`train.json`, `dev.json`, `test.json`, `v2_model.pt`, `bpe_en_25k.json`).
2. **Document Chunk Caching:** Enhanced `encode_rows` with deterministic disk caching (`.npy`) in the output directory, eliminating redundant CPU re-tokenization while preserving identical token sequences.
3. **Dual Submission Packaging:** In addition to `phase4_v2_improved_submission.zip`, generated root `submission.zip` matching Codabench competition standards, and copied it directly to `C:\Users\LENOVO\Downloads\submission.zip` to eliminate browser-mediated download corruption.
4. **Preserved Methodology:** Retained all intended V2 Improved candidate hypotheses without alteration: rank-16 LoRMA+, unfreezing layer 3 + final norm, 2-window training with logit averaging, masked mean + attention pooling, differential learning rates, early stopping (patience=3), and dev-only calibration.


## 1. Inputs and run settings

Required inputs: `train.json`, `dev.json`, `test.json`, `v2_model.pt`, and `bpe_en_25k.json`. Attach each exactly once. The notebook fails early if any required asset is missing or ambiguous. The original V2 checkpoint is loaded strictly before Phase-4 adaptation is applied.


In [ ]:
import time
import os, json, math, gc, zipfile, random, copy, hashlib
from pathlib import Path
from collections import Counter
from functools import lru_cache
import numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from torch.utils.checkpoint import checkpoint

SEED=781
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP=DEVICE.type=='cuda'
AMP_DTYPE=torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16
INPUT_ROOT=Path(os.environ.get('CJPE_INPUT_DIR', r'D:\CJPE_Phase4_V2_Local'))
OUT=Path(os.environ.get('CJPE_OUT_DIR', r'D:\CJPE_Phase4_V2_Improved_Local\outputs\phase4_v2_improved'))
OUT.mkdir(parents=True,exist_ok=True)

# Document representation: preserve the pretrained model's 512-token source length.
CHUNK_TOKENS=512
CHUNK_STRIDE=384
MAX_CHUNKS_PER_CASE=8
TRAIN_CHUNKS_PER_CASE=2

# Keep approximately the same number of windows per micro-batch as the prior run:
# 4 cases x 2 windows = 8 windows. Accumulation gives an effective batch of 16 cases.
BATCH_CASES=4
GRAD_ACCUM_STEPS=4
INFER_CHUNK_BATCH=16

# Adaptation: wider Q/V LoRMA+ adapters, one trainable encoder block, and a pooling head.
LORMA_RANK=16
LORMA_ALPHA=32
LORMA_DROPOUT=0.05
HEAD_DROPOUT=0.15
UNFREEZE_LAST_N_ENCODER_LAYERS=1

# Differential learning rates protect pretrained weights while allowing task adaptation.
LORMA_LR=5e-5
HEAD_LR=2e-4
BACKBONE_LR=1e-5
WEIGHT_DECAY=0.01
MAX_EPOCHS=8
PATIENCE=3
PROGRESS_EVERY=100
WARMUP_RATIO=0.06
MAX_GRAD_NORM=1.0

def find_one(name):
    found=sorted(p for p in INPUT_ROOT.rglob(name) if p.is_file())
    if len(found)!=1:
        alt_roots=[Path(r'D:\CJPE_Phase4_V2_Local'), Path(r'D:\phase_4_ready_to_train_final'), Path(r'D:\v2\cs781_phase1_v2\final_model'), Path(r'C:\Users\LENOVO\Downloads')]
        for alt in alt_roots:
            alt_found=sorted(p for p in alt.rglob(name) if p.is_file())
            if len(alt_found)==1: return alt_found[0]
        raise FileNotFoundError(f'Expected one {name}; found {len(found)}: {found[:8]}')
    return found[0]

ASSET={n:find_one(n) for n in ('train.json','dev.json','test.json','v2_model.pt','bpe_en_25k.json')}
print('Device:',DEVICE,'GPU:',torch.cuda.get_device_name(0) if USE_AMP else 'none')
for n,p in ASSET.items(): print(n,'->',p)
print('AMP:',USE_AMP,AMP_DTYPE if USE_AMP else 'disabled')
print('Planned training windows per case:',TRAIN_CHUNKS_PER_CASE)



## 2. Split checks and the V2 tokenizer

Training and development labels must be integer 0/1. Test rows are reduced to `id` and `text`; any extra test fields are discarded. The exact V2 BPE (the original Phase 1 byte-level tokenizer) algorithm is preserved, with a bounded cache for repeated words.

**Duplicate-record handling:** the supplied train/dev files contain exact repeated case rows. The loader keeps one copy per ID, removes only rows with identical text and label, and raises an error if an ID has conflicting content or labels.


In [ ]:

def load_cases(path,labelled):
    with Path(path).open(encoding='utf-8') as f: rows=json.load(f)
    if not isinstance(rows,list): raise TypeError(f'{path}: expected JSON list')
    result=[]; seen={}; duplicate_count=0
    for i,r in enumerate(rows):
        if not isinstance(r,dict) or 'id' not in r or 'text' not in r: raise ValueError(f'{path}: row {i} lacks id/text')
        cid=str(r['id']); text=r['text']
        if not isinstance(text,str): raise TypeError(f'{cid}: text must be string')
        label=r.get('label') if labelled else None
        if labelled and (type(label) is not int or label not in (0,1)): raise ValueError(f'{cid}: label must be integer 0/1')
        normalized={'id':cid,'text':text}
        if labelled: normalized['label']=label
        if cid in seen:
            if result[seen[cid]] != normalized:
                raise ValueError(f'{path}: conflicting duplicate id {cid} (text or label differs)')
            duplicate_count+=1
            continue
        r.clear(); r.update(normalized)
        seen[cid]=len(result); result.append(r)
    if duplicate_count: print(f'{path}: removed {duplicate_count} exact duplicate rows; kept {len(result)} unique IDs')
    return result
TRAIN_ROWS=load_cases(ASSET['train.json'],True); DEV_ROWS=load_cases(ASSET['dev.json'],True); TEST_ROWS=load_cases(ASSET['test.json'],False)
sets=[{r['id'] for r in rows} for rows in (TRAIN_ROWS,DEV_ROWS,TEST_ROWS)]
if any(sets[i]&sets[j] for i in range(3) for j in range(i+1,3)): raise ValueError('Split IDs overlap')
print('Split sizes:',*[len(x) for x in (TRAIN_ROWS,DEV_ROWS,TEST_ROWS)])
print('Train label counts:',dict(Counter(r['label'] for r in TRAIN_ROWS)))
print('Development label counts:',dict(Counter(r['label'] for r in DEV_ROWS)))
print('Test loader retains id/text only.')

with ASSET['bpe_en_25k.json'].open(encoding='utf-8') as f: bpe=json.load(f)
bpe_itos=bpe['itos']; bpe_merges=bpe['merges']; bpe_stoi={t:i for i,t in enumerate(bpe_itos)}
if bpe['vocab_size']!=25000 or len(bpe_itos)!=25000: raise ValueError('Expected V1 BPE-25k')
merge_to_id={}
for i,(left,right) in enumerate(bpe_merges): merge_to_id[(bpe_stoi[left],bpe_stoi[right])]=261+i
PAD_TOKEN_ID=bpe_stoi['<pad>']; BOS_TOKEN_ID=bpe_stoi['<s>']; EOS_TOKEN_ID=bpe_stoi['</s>']

class ByteLevelBPE25k:
    def __init__(self,itos,stoi,merge_to_id,merges):
        self.itos=itos; self.stoi=stoi; self.merge_to_id=merge_to_id; self.merges=merges; self.vocab_size=len(itos)
        self.pad_token_id=stoi['<pad>']; self.bos_token_id=stoi['<s>']; self.eos_token_id=stoi['</s>']
        self.unk_token_id=stoi['<unk>']; self.word_boundary_id=stoi['<wb>']
    @lru_cache(maxsize=100000)
    def _encode_word(self,word):
        seq=[4+b for b in word.encode('utf-8')]+[self.word_boundary_id]
        while len(seq)>1:
            best_pos=None; best_id=None
            for i in range(len(seq)-1):
                new_id=self.merge_to_id.get((seq[i],seq[i+1]))
                if new_id is not None and (best_id is None or new_id<best_id): best_id=new_id; best_pos=i
            if best_pos is None: break
            seq=seq[:best_pos]+[best_id]+seq[best_pos+2:]
        return seq
    def encode(self,text,add_bos=False,add_eos=False):
        ids=[self.bos_token_id] if add_bos else []
        for word in text.split(): ids.extend(self._encode_word(word))
        if add_eos: ids.append(self.eos_token_id)
        return ids
    def _to_bytes(self,tid):
        tok=self.itos[tid]
        if tok.startswith('<0x') and tok.endswith('>'): return bytes([int(tok[3:-1],16)])
        if tok=='<wb>': return b' '
        if tok in {'<pad>','<unk>','<s>','</s>'}: return b''
        if tok.startswith('<bpe_'):
            j=tid-261
            if 0<=j<len(self.merges):
                a,b=self.merges[j]; return self._to_bytes(self.stoi[a])+self._to_bytes(self.stoi[b])
        return b''
    def decode(self,ids):
        out=bytearray()
        for tid in ids:
            if 0<=int(tid)<self.vocab_size: out.extend(self._to_bytes(int(tid)))
        return bytes(out).decode('utf-8',errors='replace')

tokenizer=ByteLevelBPE25k(bpe_itos,bpe_stoi,merge_to_id,bpe_merges)
probe='The Supreme Court of India has jurisdiction over constitutional matters.'
decoded_probe=tokenizer.decode(tokenizer.encode(probe))
# The final <wb> token decodes to one trailing space by design.
assert decoded_probe.rstrip()==probe, repr(decoded_probe)
print('V1 tokenizer loaded; vocabulary:',tokenizer.vocab_size)



## 3. Rebuild the exact V2 architecture and load its trained weights

The next cell embeds the architecture code from the original V2 notebook (cell 26). We load the supplied checkpoint strictly before adding any Phase 4 layers. A missing or mismatched checkpoint key stops the run.


In [ ]:
# 9. V2 Encoder-Decoder Transformer
# This is our second architecture built from scratch.
# V2 uses:
# - Encoder-decoder Transformer
# - Standard Multi-Head Attention
# - Learned positional embeddings
# - LayerNorm
# - Causal decoder self-attention
# - Decoder cross-attention
# - Standard GELU FFN
# - Residual connections
# - KV cache for autoregressive generation
#
# We keep the hidden size and attention width close to V1
# so that the comparison mainly reflects the architecture.

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------
# Standard Multi-Head Attention
# ---------------------------------------------------------

class MultiHeadAttention(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        dropout=0.0
    ):
        super().__init__()

        assert n_embd % n_head == 0

        self.n_head = n_head
        self.head_dim = n_embd // n_head

        self.q_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.k_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.v_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.out_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.dropout = dropout

    def _split_heads(self, x):
        B, T, C = x.shape

        x = x.view(
            B,
            T,
            self.n_head,
            self.head_dim
        )

        return x.transpose(1, 2)

    def _merge_heads(self, x):
        B, H, T, D = x.shape

        x = x.transpose(
            1,
            2
        ).contiguous()

        return x.view(
            B,
            T,
            H * D
        )

    def forward(
        self,
        x,
        past_key_value=None,
        use_cache=False,
        causal=True,
        key_padding_mask=None
    ):
        q = self._split_heads(
            self.q_proj(x)
        )

        k = self._split_heads(
            self.k_proj(x)
        )

        v = self._split_heads(
            self.v_proj(x)
        )

        if past_key_value is not None:

            past_k, past_v = past_key_value

            k = torch.cat(
                [past_k, k],
                dim=-2
            )

            v = torch.cat(
                [past_v, v],
                dim=-2
            )

        present = None

        if use_cache:
            present = (k, v)

        # Decoder self-attention is causal.
        # Encoder self-attention is bidirectional.
        #
        # During cached generation, the current token
        # can attend to all previous cached tokens, so
        # no explicit causal mask is needed.
        if causal and past_key_value is None:

            T_q = q.size(-2)
            T_k = k.size(-2)

            causal_mask = torch.tril(
                torch.ones(
                    T_q,
                    T_k,
                    device=x.device,
                    dtype=torch.bool
                )
            )

        else:
            causal_mask = None

        if key_padding_mask is not None:
            key_mask = key_padding_mask[:, None, None, :].bool()
            if causal_mask is None:
                causal_mask = key_mask
            else:
                causal_mask = causal_mask[None, None, :, :] & key_mask

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=causal_mask,
            dropout_p=(
                self.dropout
                if self.training
                else 0.0
            ),
            is_causal=False
        )

        y = self._merge_heads(y)

        y = self.out_proj(y)

        return y, present


# ---------------------------------------------------------
# Cross Attention
# ---------------------------------------------------------

class CrossAttention(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        dropout=0.0
    ):
        super().__init__()

        assert n_embd % n_head == 0

        self.n_head = n_head
        self.head_dim = n_embd // n_head

        self.q_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.k_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.v_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.out_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.dropout = dropout

    def _split_heads(self, x):
        B, T, C = x.shape

        x = x.view(
            B,
            T,
            self.n_head,
            self.head_dim
        )

        return x.transpose(1, 2)

    def _merge_heads(self, x):
        B, H, T, D = x.shape

        x = x.transpose(
            1,
            2
        ).contiguous()

        return x.view(
            B,
            T,
            H * D
        )

    def forward(
        self,
        x,
        encoder_output
    ):
        q = self._split_heads(
            self.q_proj(x)
        )

        k = self._split_heads(
            self.k_proj(encoder_output)
        )

        v = self._split_heads(
            self.v_proj(encoder_output)
        )

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            dropout_p=(
                self.dropout
                if self.training
                else 0.0
            ),
            is_causal=False
        )

        y = self._merge_heads(y)

        y = self.out_proj(y)

        return y


# ---------------------------------------------------------
# Standard GELU Feed Forward Network
# ---------------------------------------------------------

class V2FeedForward(nn.Module):
    def __init__(
        self,
        n_embd,
        multiplier=4,
        dropout=0.0
    ):
        super().__init__()

        hidden_dim = int(
            n_embd * multiplier
        )

        self.fc1 = nn.Linear(
            n_embd,
            hidden_dim
        )

        self.fc2 = nn.Linear(
            hidden_dim,
            n_embd
        )

        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        x = self.dropout(x)

        return x


# ---------------------------------------------------------
# Encoder Layer
# ---------------------------------------------------------

class EncoderLayer(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(
            n_embd
        )

        self.self_attention = MultiHeadAttention(
            n_embd=n_embd,
            n_head=n_head,
            dropout=dropout
        )

        self.norm2 = nn.LayerNorm(
            n_embd
        )

        self.ffn = V2FeedForward(
            n_embd=n_embd,
            multiplier=ffn_multiplier,
            dropout=dropout
        )

    def forward(self, x, key_padding_mask=None):

        # Encoder self-attention is bidirectional.
        normalized_x = self.norm1(x)

        attention_output, _ = self.self_attention(
            normalized_x,
            causal=False,
            key_padding_mask=key_padding_mask
        )

        x = x + attention_output

        # Feed-forward network.
        x = x + self.ffn(
            self.norm2(x)
        )

        return x


# ---------------------------------------------------------
# Decoder Layer
# ---------------------------------------------------------

class DecoderLayer(nn.Module):
    def __init__(
        self,
        n_embd,
        n_head,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(
            n_embd
        )

        self.self_attention = MultiHeadAttention(
            n_embd=n_embd,
            n_head=n_head,
            dropout=dropout
        )

        self.norm2 = nn.LayerNorm(
            n_embd
        )

        self.cross_attention = CrossAttention(
            n_embd=n_embd,
            n_head=n_head,
            dropout=dropout
        )

        self.norm3 = nn.LayerNorm(
            n_embd
        )

        self.ffn = V2FeedForward(
            n_embd=n_embd,
            multiplier=ffn_multiplier,
            dropout=dropout
        )

    def forward(
        self,
        x,
        encoder_output,
        past_key_value=None,
        use_cache=False
    ):

        # 1. Decoder causal self-attention.
        attention_output, present = self.self_attention(
            self.norm1(x),
            past_key_value=past_key_value,
            use_cache=use_cache,
            causal=True
        )

        x = x + attention_output

        # 2. Decoder attends to the encoder output.
        cross_output = self.cross_attention(
            self.norm2(x),
            encoder_output
        )

        x = x + cross_output

        # 3. Feed-forward network.
        x = x + self.ffn(
            self.norm3(x)
        )

        return x, present


# ---------------------------------------------------------
# V2 Encoder-Decoder Legal Language Model
# ---------------------------------------------------------

class V2LegalLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        n_embd,
        n_encoder_layers,
        n_decoder_layers,
        n_head,
        source_length,
        target_length,
        ffn_multiplier,
        dropout
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.n_embd = n_embd

        self.source_length = source_length
        self.target_length = target_length

        # Shared token embedding.
        # We also tie this weight to the final LM head.
        self.token_embedding = nn.Embedding(
            vocab_size,
            n_embd
        )

        # Encoder and decoder use learned positional embeddings.
        self.encoder_position_embedding = nn.Embedding(
            source_length,
            n_embd
        )

        self.decoder_position_embedding = nn.Embedding(
            target_length,
            n_embd
        )

        # Encoder stack.
        self.encoder_layers = nn.ModuleList([
            EncoderLayer(
                n_embd=n_embd,
                n_head=n_head,
                ffn_multiplier=ffn_multiplier,
                dropout=dropout
            )
            for _ in range(n_encoder_layers)
        ])

        # Decoder stack.
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(
                n_embd=n_embd,
                n_head=n_head,
                ffn_multiplier=ffn_multiplier,
                dropout=dropout
            )
            for _ in range(n_decoder_layers)
        ])

        self.encoder_final_norm = nn.LayerNorm(
            n_embd
        )

        self.decoder_final_norm = nn.LayerNorm(
            n_embd
        )

        # Tied language-model head.
        self.lm_head = nn.Linear(
            n_embd,
            vocab_size,
            bias=False
        )

        self.lm_head.weight = (
            self.token_embedding.weight
        )

        self.apply(
            self._init_weights
        )

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

            if module.bias is not None:
                nn.init.zeros_(
                    module.bias
                )

        elif isinstance(module, nn.Embedding):

            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02
            )

    def encode(
        self,
        encoder_input_ids,
        attention_mask=None
    ):

        B, T = encoder_input_ids.shape
        if attention_mask is None:
            attention_mask = torch.ones_like(encoder_input_ids, dtype=torch.bool)
        else:
            attention_mask = attention_mask.bool()

        positions = torch.arange(
            T,
            device=encoder_input_ids.device
        )

        x = self.token_embedding(
            encoder_input_ids
        )

        x = x + self.encoder_position_embedding(
            positions
        ).unsqueeze(0)

        for layer in self.encoder_layers:
            x = layer(x, key_padding_mask=attention_mask)

        x = self.encoder_final_norm(x)

        return x

    def decode(
        self,
        decoder_input_ids,
        encoder_output,
        past_key_values=None,
        use_cache=False
    ):

        B, T = decoder_input_ids.shape

        if past_key_values is None:

            position_offset = 0

        else:

            position_offset = (
                past_key_values[0][0].size(-2)
            )

        positions = torch.arange(
            position_offset,
            position_offset + T,
            device=decoder_input_ids.device
        )

        x = self.token_embedding(
            decoder_input_ids
        )

        x = x + self.decoder_position_embedding(
            positions
        ).unsqueeze(0)

        presents = [] if use_cache else None

        for layer_index, layer in enumerate(
            self.decoder_layers
        ):

            past = None

            if past_key_values is not None:

                past = past_key_values[
                    layer_index
                ]

            x, present = layer(
                x,
                encoder_output,
                past_key_value=past,
                use_cache=use_cache
            )

            if use_cache:

                presents.append(
                    present
                )

        x = self.decoder_final_norm(x)

        logits = self.lm_head(x)

        return logits, presents

    def forward(
        self,
        encoder_input_ids,
        decoder_input_ids,
        labels=None,
        use_cache=False,
        past_key_values=None,
        encoder_output=None
    ):

        # Encode the source sequence only once.
        if encoder_output is None:

            encoder_output = self.encode(
                encoder_input_ids
            )

        logits, presents = self.decode(
            decoder_input_ids,
            encoder_output,
            past_key_values=past_key_values,
            use_cache=use_cache
        )

        loss = None

        if labels is not None:

            loss = F.cross_entropy(
                logits.reshape(
                    -1,
                    logits.size(-1)
                ),
                labels.reshape(-1),
                ignore_index=PAD_TOKEN_ID
            )

        return (
            logits,
            loss,
            presents,
            encoder_output
        )


print("V2 Encoder-Decoder model classes defined.")

# The classifier uses `encode(input_ids, attention_mask)`; the decoder remains
# available during strict checkpoint loading, then is released from GPU memory.


In [ ]:
# V2 Phase 1 architecture and checkpoint metadata from the original V2 notebook.
N_EMBD=768; N_ENCODER_LAYERS=4; N_DECODER_LAYERS=5; N_HEAD=12
SOURCE_LENGTH=512; TARGET_LENGTH=512; FFN_MULTIPLIER=4; DROPOUT=0.0
V2_CONFIG={'architecture':'encoder_decoder','vocab_size':tokenizer.vocab_size,'n_embd':N_EMBD,
           'n_encoder_layers':N_ENCODER_LAYERS,'n_decoder_layers':N_DECODER_LAYERS,
           'n_head':N_HEAD,'source_length':SOURCE_LENGTH,'target_length':TARGET_LENGTH,
           'ffn_multiplier':FFN_MULTIPLIER,'dropout':DROPOUT}
v2=V2LegalLM(**{k:v for k,v in V2_CONFIG.items() if k!='architecture'})
ckpt=torch.load(ASSET['v2_model.pt'],map_location='cpu',weights_only=False)
if ckpt.get('config') != V2_CONFIG: raise ValueError(f'V2 checkpoint config mismatch: {ckpt.get("config")}')
state=ckpt.get('model_state_dict',ckpt)
missing,unexpected=v2.load_state_dict(state,strict=True)
assert not missing and not unexpected
PRETRAINED_TOTAL_PARAMS=sum(p.numel() for p in v2.parameters())
print('Strict V2 checkpoint load passed; full checkpoint parameters:',f'{PRETRAINED_TOTAL_PARAMS:,}')
print('Phase 1 best validation loss:',ckpt.get('best_val_loss'),'perplexity:',ckpt.get('best_val_perplexity'))
for p in v2.parameters(): p.requires_grad_(False)
# This is a classifier over the V2 source encoder; the autoregressive decoder
# and its positional/final-normalization modules are not on the classification path.
del v2.decoder_layers, v2.decoder_position_embedding, v2.decoder_final_norm, v2.lm_head
v2.to(DEVICE).eval()
ACTIVE_BACKBONE_PARAMS=sum(p.numel() for p in v2.parameters())
print('Retained frozen encoder plus shared token embeddings:',f'{ACTIVE_BACKBONE_PARAMS:,}')


## 4. LoRMA+ and selective final-layer fine-tuning

The Phase-1 V2 checkpoint is an encoder-decoder language model. For classification, this stage uses the bidirectional source encoder; the decoder is not used to generate text. LoRMA+ adapters are inserted into every encoder block's query and value projections.

A low-rank multiplicative update is applied to the frozen projection output, with zero-initialized up-projection so the adapter starts as an identity transformation. This revision increases adapter rank from 8 to 16. It also unfreezes the final encoder block's non-Q/V parameters and the encoder final normalization, while keeping pretrained Q/V projection weights frozen underneath their adapters. The new and pretrained parameter groups receive different learning rates.

This is a deliberate capacity increase, not full-model fine-tuning. It tests whether a small amount of encoder adaptation helps the binary task without allowing all pretrained parameters to drift.

The encoder receives a boolean key-padding mask. The classifier uses both masked mean pooling and learned attention pooling over non-padding token states. A smoke-test cell follows the model definition to check padding invariance, output shape, and gradient flow before training.


In [ ]:
class LoRMALinear(nn.Module):
    """LoRMA+ adapter on a frozen square projection output.

    z = W0 x; z' = z + (alpha / rank) * B(A(dropout(z))).
    B starts at zero so inserting the adapter preserves the pretrained function.
    """
    def __init__(self,base,rank=16,alpha=32,dropout=0.05):
        super().__init__()
        if base.in_features != base.out_features:
            raise ValueError('This LoRMA+ implementation expects square projections.')
        self.base=base
        self.rank=int(rank)
        self.scale=float(alpha)/float(rank)
        self.dropout=nn.Dropout(float(dropout))
        d=base.out_features
        self.A=nn.Parameter(torch.empty(rank,d,device=base.weight.device,dtype=base.weight.dtype))
        self.B=nn.Parameter(torch.zeros(d,rank,device=base.weight.device,dtype=base.weight.dtype))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))
        for p in self.base.parameters():
            p.requires_grad_(False)

    def forward(self,x):
        z=self.base(x)
        return z + F.linear(F.linear(self.dropout(z),self.A),self.B)*self.scale

# Load strictly before this point, then adapt Q/V projections in all encoder layers.
for layer in v2.encoder_layers:
    attn=layer.self_attention
    attn.q_proj=LoRMALinear(attn.q_proj,LORMA_RANK,LORMA_ALPHA,LORMA_DROPOUT)
    attn.v_proj=LoRMALinear(attn.v_proj,LORMA_RANK,LORMA_ALPHA,LORMA_DROPOUT)

# Selective fine-tuning: train the final encoder block, except its pretrained Q/V weights.
# These Q/V maps remain frozen; their LoRMA A/B matrices are trainable.
if not (1 <= UNFREEZE_LAST_N_ENCODER_LAYERS <= len(v2.encoder_layers)):
    raise ValueError('UNFREEZE_LAST_N_ENCODER_LAYERS must be between 1 and the encoder depth.')
for layer in list(v2.encoder_layers)[-UNFREEZE_LAST_N_ENCODER_LAYERS:]:
    for p in layer.parameters():
        p.requires_grad_(True)
    for projection in (layer.self_attention.q_proj, layer.self_attention.v_proj):
        for p in projection.base.parameters():
            p.requires_grad_(False)
for p in v2.encoder_final_norm.parameters():
    p.requires_grad_(True)

class CaseClassifier(nn.Module):
    """Mean + learned-attention pooling, with one case-level binary classifier."""
    def __init__(self,backbone):
        super().__init__()
        self.backbone=backbone
        self.pool_score=nn.Linear(N_EMBD,1,bias=False)
        self.head=nn.Sequential(
            nn.Dropout(HEAD_DROPOUT),
            nn.Linear(2*N_EMBD,2)
        )

    def forward(self,input_ids,attention_mask):
        mask=attention_mask.bool()
        x=self.backbone.encode(input_ids,attention_mask=mask)
        weights_mask=mask.unsqueeze(-1).to(x.dtype)

        # Broad summary: average only the real (non-padding) token states.
        mean_pooled=(x*weights_mask).sum(dim=1)/weights_mask.sum(dim=1).clamp_min(1.0)

        # Salience summary: learn a token distribution, excluding padding before softmax.
        attn_scores=self.pool_score(x).squeeze(-1).float()
        attn_scores=attn_scores.masked_fill(~mask,torch.finfo(attn_scores.dtype).min)
        token_weights=torch.softmax(attn_scores,dim=1).to(dtype=x.dtype)
        attn_pooled=(x*token_weights.unsqueeze(-1)).sum(dim=1)

        pooled=torch.cat([mean_pooled,attn_pooled],dim=-1)
        return self.head(pooled)

model=CaseClassifier(v2).to(DEVICE)
trainable_params=[(n,p) for n,p in model.named_parameters() if p.requires_grad]
trainable=sum(p.numel() for _,p in trainable_params)
active_total=sum(p.numel() for p in model.parameters())
adapter_count=sum(p.numel() for n,p in trainable_params if n.endswith(('.A','.B')))
pool_head_count=sum(p.numel() for n,p in trainable_params if n.startswith(('pool_score.','head.')))
backbone_count=trainable-adapter_count-pool_head_count
print(f'Trainable parameters: {trainable:,} / active parameters: {active_total:,} / full checkpoint: {PRETRAINED_TOTAL_PARAMS:,}')
print(f'Trainable breakdown — LoRMA+ adapters: {adapter_count:,}; pooling/classification head: {pool_head_count:,}; final-block backbone: {backbone_count:,}')
assert adapter_count == 8*2*N_EMBD*LORMA_RANK, 'Unexpected LoRMA+ adapter count'
assert pool_head_count > 0 and backbone_count > 0, 'Expected a trainable head and final encoder block'
assert trainable < 10_000_000, 'Trainable parameter budget unexpectedly large; inspect unfreezing configuration'


### 4.1. Pre-training smoke tests

These low-cost checks are run before the full training loop. They do not estimate task performance; they test implementation properties: logits have the right shape, changing token IDs in masked padding does not change real-token encoder states, and gradients for adapters, the head, and the selectively unfrozen block are finite.


In [ ]:
_smoke_ids=torch.randint(0,tokenizer.vocab_size-1,(2,32),device=DEVICE)
_smoke_ids=_smoke_ids+(_smoke_ids>=PAD_TOKEN_ID).long()  # exclude PAD from positions marked valid
_smoke_mask=torch.ones_like(_smoke_ids,dtype=torch.bool)
_smoke_mask[:,24:]=False
_smoke_ids[~_smoke_mask]=PAD_TOKEN_ID
_smoke_changed=_smoke_ids.clone()
_smoke_changed[~_smoke_mask]=(PAD_TOKEN_ID+7)%tokenizer.vocab_size

model.eval()
with torch.no_grad():
    _enc_a=model.backbone.encode(_smoke_ids,_smoke_mask)
    _enc_b=model.backbone.encode(_smoke_changed,_smoke_mask)
    torch.testing.assert_close(_enc_a[_smoke_mask],_enc_b[_smoke_mask],rtol=1e-4,atol=1e-5)
    _smoke_logits=model(_smoke_ids,_smoke_mask)
    assert _smoke_logits.shape==(2,2), f'Unexpected logits shape: {_smoke_logits.shape}'
    assert torch.isfinite(_smoke_logits).all(), 'Non-finite logits in smoke test'

# Exercise the same [cases, windows, tokens] flatten/aggregate shape used by training.
_smoke_multi_ids=torch.stack([_smoke_ids,_smoke_changed],dim=1)
_smoke_multi_mask=torch.stack([_smoke_mask,_smoke_mask],dim=1)
model.train()
_mw_case_count,_mw_window_count,_mw_seq_len=_smoke_multi_ids.shape
_mw_logits=model(
    _smoke_multi_ids.reshape(_mw_case_count*_mw_window_count,_mw_seq_len),
    _smoke_multi_mask.reshape(_mw_case_count*_mw_window_count,_mw_seq_len)
)
_mw_case_logits=_mw_logits.reshape(_mw_case_count,_mw_window_count,2).mean(dim=1)
assert _mw_case_logits.shape==(2,2), f'Unexpected aggregated logit shape: {_mw_case_logits.shape}'
_smoke_labels=torch.tensor([0,1],device=DEVICE,dtype=torch.long)
_grad_loss=F.cross_entropy(_mw_case_logits,_smoke_labels)
assert torch.isfinite(_grad_loss), 'Non-finite smoke-test loss'
_grad_loss.backward()
_required_grad_params=[
    model.backbone.encoder_layers[0].self_attention.q_proj.B,
    model.pool_score.weight,
    model.head[1].weight,
    model.backbone.encoder_layers[-1].ffn.fc1.weight
]
for _p in _required_grad_params:
    assert _p.grad is not None, 'Expected a gradient was not produced'
    assert torch.isfinite(_p.grad).all(), 'Non-finite gradient in smoke test'
model.zero_grad(set_to_none=True)
model.eval()
print('Smoke tests passed: output shape, two-window aggregation, finite loss/gradients, and padding-mask invariance.')
print('The smoke test did not train the model or measure development performance.')


## 5. Long judgments: overlapping windows with multi-window training

The encoder remains limited to 512 source tokens. We use 512-token windows with stride 384 and retain up to eight windows spread across each case. Unlike the previous run, training samples **two windows per case per epoch** and computes the loss after averaging their logits into a single case-level prediction. The sampled windows change reproducibly each epoch; when a case has only one available window, it is duplicated to keep tensor shapes consistent.

Development and test inference score every selected window and average the window logits per case. Training and inference therefore both use case-level aggregation, although inference has access to more windows. This improves alignment while keeping GPU memory close to the old eight-window-per-microbatch setup (four cases times two windows). It does not guarantee that every source token is retained: the cap of eight windows is deliberately unchanged.


In [ ]:
def select_windows(text):
    ids=tokenizer.encode(text,add_bos=True)
    if not ids: ids=[BOS_TOKEN_ID]
    if len(ids)<=CHUNK_TOKENS:
        starts=[0]
    else:
        all_starts=list(range(0,len(ids)-CHUNK_TOKENS+1,CHUNK_STRIDE))
        if all_starts[-1]!=len(ids)-CHUNK_TOKENS:
            all_starts.append(len(ids)-CHUNK_TOKENS)
        take=min(MAX_CHUNKS_PER_CASE,len(all_starts))
        starts_idx=sorted(set(np.linspace(0,len(all_starts)-1,take).round().astype(int).tolist()))
        starts=[all_starts[i] for i in starts_idx]
    chunks=[]
    for start in starts:
        x=ids[start:start+CHUNK_TOKENS]
        chunks.append(x+[PAD_TOKEN_ID]*(CHUNK_TOKENS-len(x)))
    return np.asarray(chunks,dtype=np.int32)

def encode_rows(rows,name):
    cache_file = OUT / 'cache' / f'{name}_windows.npy'
    if cache_file.exists():
        print(f'Loading precomputed {name} windows from {cache_file}...')
        cached = np.load(cache_file, allow_pickle=True)
        if len(cached) == len(rows):
            counts = [len(c) for c in cached]
            print(f'{name} windows: {sum(counts)} documents: {len(rows)} median windows: {int(np.median(counts))}')
            return cached
    counts=[]; chunks=[]
    for i,row in enumerate(rows):
        c=select_windows(row['text']); chunks.append(c); counts.append(len(c))
        if (i+1)%5000==0: print(name,i+1,'/',len(rows))
    print(name,'windows:',sum(counts),'documents:',len(rows),'median windows:',int(np.median(counts)))
    cache_file.parent.mkdir(parents=True, exist_ok=True)
    np.save(cache_file, np.asarray(chunks, dtype=object))
    return chunks

TRAIN_WINDOWS=encode_rows(TRAIN_ROWS,'train')
DEV_WINDOWS=encode_rows(DEV_ROWS,'dev')
TEST_WINDOWS=encode_rows(TEST_ROWS,'test')

class TrainDataset(Dataset):
    def __init__(self,rows,windows):
        self.rows=rows
        self.windows=windows
        self.epoch=0
    def set_epoch(self,e):
        self.epoch=int(e)
    def __len__(self):
        return len(self.rows)
    def __getitem__(self,i):
        ws=self.windows[i]
        # Deterministic per case and epoch; sampling without replacement when possible.
        rng=np.random.default_rng(SEED+100_003*self.epoch+i)
        replace=len(ws)<TRAIN_CHUNKS_PER_CASE
        chosen=rng.choice(len(ws),size=TRAIN_CHUNKS_PER_CASE,replace=replace)
        x=ws[chosen]
        mask=(x!=PAD_TOKEN_ID)
        return torch.tensor(x,dtype=torch.long),torch.tensor(mask,dtype=torch.bool),self.rows[i]['label']

class EvalDataset(Dataset):
    def __init__(self,rows,windows):
        self.rows=rows; self.windows=windows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self,i):
        return self.windows[i],self.rows[i]['id'],self.rows[i].get('label')

train_ds=TrainDataset(TRAIN_ROWS,TRAIN_WINDOWS)
dev_ds=EvalDataset(DEV_ROWS,DEV_WINDOWS)
test_ds=EvalDataset(TEST_ROWS,TEST_WINDOWS)
train_loader=DataLoader(train_ds,batch_size=BATCH_CASES,shuffle=True,num_workers=0,pin_memory=USE_AMP)


## 6. Train with differentiated learning rates and case-level multi-window loss

The loss is calculated at the case level: the logits from two sampled windows are averaged before weighted cross-entropy. The optimizer has separate learning rates for LoRMA+ adapters, the new pooling/classification head, and the selectively unfrozen final encoder block. Pretrained parameters receive the smallest learning rate to limit catastrophic drift. Training uses the original train/dev split, training-label-derived class weights, gradient clipping, warmup plus linear decay, and early stopping on development Macro-F1.

The epoch limit rises from four to eight because the previous best occurred at epoch two and the score fluctuated thereafter. Eight is a maximum; patience-based early stopping prevents simply running extra epochs after development performance stops improving.


In [ ]:
if DEVICE.type!='cuda':
    raise RuntimeError('Select a GPU accelerator before training the V2 checkpoint.')

def scores(y_true,y_pred):
    out={}; f1s=[]
    for c in (0,1):
        tp=sum(a==c and b==c for a,b in zip(y_true,y_pred))
        fp=sum(a!=c and b==c for a,b in zip(y_true,y_pred))
        fn=sum(a==c and b!=c for a,b in zip(y_true,y_pred))
        p=tp/(tp+fp) if tp+fp else 0.
        r=tp/(tp+fn) if tp+fn else 0.
        f=2*p*r/(p+r) if p+r else 0.
        out[str(c)]={'precision':p,'recall':r,'f1':f,'support':sum(a==c for a in y_true)}
        f1s.append(f)
    out['macro_f1']=sum(f1s)/2
    out['accuracy']=sum(a==b for a,b in zip(y_true,y_pred))/max(1,len(y_true))
    return out

@torch.no_grad()
def predict_windows(rows,windows,mode='all',threshold=0.5):
    model.eval(); selected=[]; owners=[]
    for i,ws in enumerate(windows):
        if mode=='begin':
            chosen=ws[:1]
        elif mode=='middle':
            chosen=ws[(len(ws)-1)//2:(len(ws)-1)//2+1]
        elif mode=='end':
            chosen=ws[-1:]
        else:
            chosen=ws
        selected.extend(chosen); owners.extend([i]*len(chosen))
    flat=np.stack(selected,axis=0)
    if flat.ndim!=2:
        raise ValueError(f'Expected prediction windows with shape [windows, tokens], got {flat.shape}')
    all_logits=[]
    infer_started=time.time()
    infer_total=math.ceil(len(flat)/INFER_CHUNK_BATCH)
    for batch_idx,j in enumerate(range(0,len(flat),INFER_CHUNK_BATCH),start=1):
        ids=torch.as_tensor(flat[j:j+INFER_CHUNK_BATCH],dtype=torch.long,device=DEVICE)
        mask=ids.ne(PAD_TOKEN_ID)
        with torch.autocast(device_type='cuda',dtype=AMP_DTYPE,enabled=USE_AMP):
            z=model(ids,mask)
        all_logits.append(z.float().cpu().numpy())
        if batch_idx==1 or batch_idx%PROGRESS_EVERY==0 or batch_idx==infer_total:
            elapsed=time.time()-infer_started
            rate=batch_idx/max(elapsed,1e-6)
            eta=(infer_total-batch_idx)/max(rate,1e-6)
            print(f'Predict {mode}: batch {batch_idx}/{infer_total} | elapsed {elapsed/60:.1f} min | ETA {eta/60:.1f} min',flush=True)
    window_logits=np.concatenate(all_logits,axis=0)
    logits_sum=np.zeros((len(rows),2),dtype=np.float64)
    counts=np.zeros(len(rows),dtype=np.int64)
    np.add.at(logits_sum,np.asarray(owners),window_logits)
    np.add.at(counts,np.asarray(owners),1)
    logits=(logits_sum/counts[:,None]).astype(np.float32)
    probs=torch.softmax(torch.tensor(logits),dim=-1).numpy()[:,1]
    return logits,probs,(probs>=threshold).astype(int).tolist()

counts=np.bincount([r['label'] for r in TRAIN_ROWS],minlength=2)
class_weights=torch.tensor(len(TRAIN_ROWS)/(2*np.maximum(counts,1)),dtype=torch.float,device=DEVICE)
criterion=nn.CrossEntropyLoss(weight=class_weights)

adapter_params=[p for n,p in model.named_parameters() if p.requires_grad and n.endswith(('.A','.B'))]
head_params=[p for n,p in model.named_parameters() if p.requires_grad and n.startswith(('pool_score.','head.'))]
backbone_params=[p for n,p in model.named_parameters()
                 if p.requires_grad and not n.endswith(('.A','.B')) and not n.startswith(('pool_score.','head.'))]
trainable_ids={id(p) for p in model.parameters() if p.requires_grad}
optimizer_ids=[id(p) for group in (adapter_params,head_params,backbone_params) for p in group]
assert len(optimizer_ids)==len(set(optimizer_ids)), 'A trainable parameter is assigned to more than one optimizer group'
assert set(optimizer_ids)==trainable_ids, 'Some trainable parameters are missing from the optimizer'
assert adapter_params and head_params and backbone_params, 'Expected adapter, head, and backbone optimizer groups'

opt=torch.optim.AdamW([
    {'params':adapter_params,'lr':LORMA_LR},
    {'params':head_params,'lr':HEAD_LR},
    {'params':backbone_params,'lr':BACKBONE_LR},
],weight_decay=WEIGHT_DECAY)

total_updates=math.ceil(len(train_loader)/GRAD_ACCUM_STEPS)*MAX_EPOCHS
warmup=max(1,int(total_updates*WARMUP_RATIO))
def lr_scale(step):
    if step<warmup:
        return min((step+1)/warmup,1.)
    return max(0.,(total_updates-step)/(total_updates-warmup))
scheduler=torch.optim.lr_scheduler.LambdaLR(opt,lr_scale)
scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP and AMP_DTYPE==torch.float16)

history=[]; best_f1=-1.; best_loss=float('inf'); best_epoch=0; stale=0; update=0
for epoch in range(MAX_EPOCHS):
    train_ds.set_epoch(epoch)
    model.train()
    opt.zero_grad(set_to_none=True)
    loss_sum=0.; seen=0; pending=0; epoch_started=time.time()
    epoch_lrs=[float(g['lr']) for g in opt.param_groups]
    print(f'Starting epoch {epoch+1}/{MAX_EPOCHS}: {len(train_loader)} batches; sampled windows/case={TRAIN_CHUNKS_PER_CASE}',flush=True)

    for step,(ids,mask,y) in enumerate(train_loader):
        # Dataset tensors: [case_batch, windows_per_case, tokens].
        ids=ids.to(DEVICE,non_blocking=True)
        mask=mask.to(DEVICE,non_blocking=True)
        y=torch.as_tensor(y,device=DEVICE,dtype=torch.long)
        batch_size,windows_per_case,seq_len=ids.shape
        flat_ids=ids.reshape(batch_size*windows_per_case,seq_len)
        flat_mask=mask.reshape(batch_size*windows_per_case,seq_len)

        with torch.autocast(device_type='cuda',dtype=AMP_DTYPE,enabled=USE_AMP):
            window_logits=model(flat_ids,flat_mask)
            case_logits=window_logits.reshape(batch_size,windows_per_case,2).mean(dim=1)
            loss=criterion(case_logits,y)

        if not torch.isfinite(loss):
            raise FloatingPointError(f'Non-finite training loss at epoch {epoch+1}, batch {step+1}')
        scaler.scale(loss/GRAD_ACCUM_STEPS).backward()
        pending+=1
        loss_sum+=float(loss.detach())*len(y)
        seen+=len(y)

        if step==0 or (step+1)%PROGRESS_EVERY==0 or step+1==len(train_loader):
            elapsed=time.time()-epoch_started
            rate=(step+1)/max(elapsed,1e-6)
            eta=(len(train_loader)-step-1)/max(rate,1e-6)
            print(f'Train epoch {epoch+1}/{MAX_EPOCHS}: batch {step+1}/{len(train_loader)} | running loss {loss_sum/max(seen,1):.4f} | elapsed {elapsed/60:.1f} min | ETA {eta/60:.1f} min',flush=True)

        if pending==GRAD_ACCUM_STEPS or step==len(train_loader)-1:
            if pending<GRAD_ACCUM_STEPS:
                correction=GRAD_ACCUM_STEPS/pending
                for p in model.parameters():
                    if p.grad is not None:
                        p.grad.mul_(correction)
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],MAX_GRAD_NORM)
            scaler.step(opt); scaler.update()
            opt.zero_grad(set_to_none=True)
            scheduler.step(); update+=1; pending=0

    dev_logits,dev_probs,dev_pred=predict_windows(DEV_ROWS,DEV_WINDOWS)
    dev_y=[r['label'] for r in DEV_ROWS]
    dev_metrics=scores(dev_y,dev_pred)
    record={
        'epoch':epoch+1,
        'train_loss':loss_sum/max(seen,1),
        'dev':dev_metrics,
        'lr_lorma_start':epoch_lrs[0],
        'lr_head_start':epoch_lrs[1],
        'lr_backbone_start':epoch_lrs[2],
        'optimizer_updates':update,
        'train_windows_per_case':TRAIN_CHUNKS_PER_CASE
    }
    history.append(record)
    print(f"Epoch {epoch+1}/{MAX_EPOCHS} | train loss {record['train_loss']:.4f} | dev macro-F1 {dev_metrics['macro_f1']:.4f} | accuracy {dev_metrics['accuracy']:.4f}")

    if dev_metrics['macro_f1']>best_f1 or (dev_metrics['macro_f1']==best_f1 and record['train_loss']<best_loss):
        best_f1=dev_metrics['macro_f1']; best_loss=record['train_loss']; best_epoch=epoch+1; stale=0
        trainable_names={n for n,p in model.named_parameters() if p.requires_grad}
        torch.save({
            'trainable_state_dict':{k:v.detach().cpu() for k,v in model.state_dict().items() if k in trainable_names},
            'epoch':best_epoch,
            'dev_macro_f1_at_050':best_f1,
            'train_loss':best_loss,
            'seed':SEED,
            'config':{
                'method':'LoRMA+ with selective final-block fine-tuning',
                'rank':LORMA_RANK,
                'alpha':LORMA_ALPHA,
                'unfreeze_last_n_encoder_layers':UNFREEZE_LAST_N_ENCODER_LAYERS,
                'train_chunks_per_case':TRAIN_CHUNKS_PER_CASE,
                'chunk_tokens':CHUNK_TOKENS,
                'chunk_stride':CHUNK_STRIDE,
                'max_chunks_per_case':MAX_CHUNKS_PER_CASE,
                'pooling':'masked mean + learned attention',
                'head_dropout':HEAD_DROPOUT
            }
        },OUT/'v2_improved_best.pt')
    else:
        stale+=1

    (OUT/'training_history.json').write_text(json.dumps(history,indent=2),encoding='utf-8')
    if stale>=PATIENCE:
        print('Early stopping: development Macro-F1 did not improve.')
        break

print('Best epoch:',best_epoch,'best raw dev Macro-F1 at threshold 0.50:',best_f1)
print('Training updates:',update)


## 7. Restore the best checkpoint, inspect context ablations, and calibrate on development only

Checkpoint selection uses the uncalibrated development Macro-F1 at threshold 0.50 for each epoch. Once the best checkpoint is restored, the notebook reports beginning-only, middle-only, end-only, and all-window diagnostics. It then searches thresholds on the development probabilities only and freezes the selected threshold for test inference. The test set is never used to select the epoch or threshold.


In [ ]:
best=torch.load(OUT/'v2_improved_best.pt',map_location=DEVICE,weights_only=False)
own=model.state_dict()
saved_state=best['trainable_state_dict']
unknown_keys=set(saved_state)-set(own)
if unknown_keys:
    raise KeyError(f'Checkpoint contains unknown trainable keys: {sorted(unknown_keys)[:10]}')
own.update(saved_state)
model.load_state_dict(own,strict=True)
model.eval()

dev_y=[r['label'] for r in DEV_ROWS]
ablations={}
for mode in ('begin','middle','end','all'):
    _,prob,pred=predict_windows(DEV_ROWS,DEV_WINDOWS,mode=mode)
    ablations[mode]=scores(dev_y,pred)
    print(mode,ablations[mode])

_,dev_prob,dev_pred=predict_windows(DEV_ROWS,DEV_WINDOWS)
thresholds=np.arange(0.30,0.701,0.01)
threshold_results=[]
for t in thresholds:
    m=scores(dev_y,(dev_prob>=t).astype(int).tolist())
    threshold_results.append((m['macro_f1'],t,m))
_,best_threshold,best_calibrated=max(threshold_results,key=lambda x:(x[0],-abs(x[1]-0.5)))
print('Development-selected threshold:',round(float(best_threshold),2),'metrics:',best_calibrated)

summary={
    'experiment_id':'v2_lorma_plus_selective_ft_multwindow_attention_v1',
    'baseline_reference':{'dev_macro_f1_at_threshold_0_50':0.5838,'note':'Prior reported run; compare only if split/preprocessing are identical.'},
    'best_epoch':best_epoch,
    'best_checkpoint_selection_macro_f1_at_050':best_f1,
    'development_threshold':float(best_threshold),
    'development_calibrated_metrics':best_calibrated,
    'context_ablation':ablations,
    'class_weights':class_weights.cpu().tolist(),
    'trainable_parameters':trainable,
    'adapter_parameters':adapter_count,
    'pooling_and_head_parameters':pool_head_count,
    'selectively_unfrozen_backbone_parameters':backbone_count,
    'active_model_parameters':active_total,
    'full_pretrained_parameters':PRETRAINED_TOTAL_PARAMS,
    'revision_notes':[
        'LoRMA+ rank increased from 8 to 16; alpha scaled from 16 to 32 to preserve alpha/rank=2.',
        'Final encoder block and encoder final normalization unfrozen; pretrained Q/V projection weights remain frozen.',
        'Training uses two reproducibly sampled windows per case and averages window logits before case-level loss.',
        'Classifier uses masked mean pooling concatenated with learned attention pooling.',
        'Differential learning rates and up to eight epochs with patience-based early stopping.',
        'Smoke tests check finite logits/gradients, output shape, and padding-mask invariance.'
    ],
    'configuration':{
        'seed':SEED,'chunk_tokens':CHUNK_TOKENS,'chunk_stride':CHUNK_STRIDE,
        'max_chunks_per_case':MAX_CHUNKS_PER_CASE,'train_chunks_per_case':TRAIN_CHUNKS_PER_CASE,
        'lorma_rank':LORMA_RANK,'lorma_alpha':LORMA_ALPHA,
        'unfreeze_last_n_encoder_layers':UNFREEZE_LAST_N_ENCODER_LAYERS,
        'lorma_lr':LORMA_LR,'head_lr':HEAD_LR,'backbone_lr':BACKBONE_LR,
        'max_epochs':MAX_EPOCHS,'patience':PATIENCE,'head_dropout':HEAD_DROPOUT,
        'pooling':'masked mean + learned attention'
    }
}
(OUT/'development_metrics.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')


## 8. Predict test cases and create the required submission ZIP

The test loader retains only `id` and `text`; labels or other extra fields are discarded. There is one integer `pred` in `{0,1}` per original test ID. The output ZIP contains exactly one root-level `prediction.json`. The file is independently checked after writing.


In [ ]:
test_logits,test_prob,test_pred=predict_windows(TEST_ROWS,TEST_WINDOWS,threshold=float(best_threshold))
prediction=[{'id':row['id'],'pred':int(pred)} for row,pred in zip(TEST_ROWS,test_pred)]
assert len(prediction)==len(TEST_ROWS)
assert [x['id'] for x in prediction]==[x['id'] for x in TEST_ROWS]
assert len({x['id'] for x in prediction})==len(prediction)
assert all(type(x['pred']) is int and x['pred'] in (0,1) for x in prediction)

pred_path=OUT/'prediction.json'
pred_path.write_text(json.dumps(prediction,ensure_ascii=False,indent=2),encoding='utf-8')

import csv
prob_path=OUT/'test_probabilities.csv'
with prob_path.open('w',encoding='utf-8',newline='') as f:
    writer=csv.writer(f)
    writer.writerow(['id','prob_class_1','pred'])
    for row,prob,pred in zip(TEST_ROWS,test_prob,test_pred):
        writer.writerow([row['id'],float(prob),int(pred)])

zip_path=OUT/'phase4_v2_improved_submission.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path,arcname='prediction.json')
with zipfile.ZipFile(zip_path) as z:
    assert z.namelist()==['prediction.json']
    assert z.testzip() is None
    roundtrip=json.loads(z.read('prediction.json'))
assert roundtrip==prediction

# Also write standard submission.zip for Codabench
zip_path_std=OUT/'submission.zip'
with zipfile.ZipFile(zip_path_std,'w',compression=zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path,arcname='prediction.json')

import shutil
dl_dir = Path(r'C:\Users\LENOVO\Downloads')
if dl_dir.exists():
    shutil.copy2(zip_path_std, dl_dir / 'submission.zip')
    shutil.copy2(zip_path, dl_dir / 'phase4_v2_improved_submission.zip')
    print('Copied submission.zip and phase4_v2_improved_submission.zip directly to Downloads.')

artifact_hash=hashlib.sha256(zip_path.read_bytes()).hexdigest()
(OUT/'submission_validation.json').write_text(json.dumps({
    'zip_path':str(zip_path),
    'sha256':artifact_hash,
    'zip_entries':['prediction.json'],
    'crc_check':'passed',
    'prediction_count':len(prediction),
    'ids_unique':len({x['id'] for x in prediction})==len(prediction),
    'ids_match_test_order':[x['id'] for x in prediction]==[x['id'] for x in TEST_ROWS],
    'labels_are_integer_0_or_1':all(type(x['pred']) is int and x['pred'] in (0,1) for x in prediction)
},indent=2),encoding='utf-8')

print('Created:',pred_path,'and',zip_path)
print('Submission SHA-256:',artifact_hash)
print('Test predictions:',len(prediction),'predicted class counts:',dict(Counter(x['pred'] for x in prediction)))
print('Development metrics:',json.dumps(summary,indent=2))


## What this stage does and does not establish

This notebook runs one improved V2 Phase-4 candidate, not the complete model-comparison program. It does not by itself cover V3, V4, or the knowledge-distillation experiment. Its purpose is to test whether modestly increased adaptation capacity, multi-window case-level training, and attention pooling improve held-out development Macro-F1 relative to the reported 0.5838 baseline.

No outcome score is guaranteed. Record the actual runtime, best epoch, raw and calibrated development Macro-F1, class-wise precision/recall, context-ablation findings, checkpoint hash, and any competition score before accepting the change. If the revised run fails to improve development performance, revert individual changes or run controlled ablations instead of assuming the larger setup is better.
